# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [17]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
sys.path.append('..')
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [18]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROK_API_KEY')

if api_key and api_key[:3] == 'xai':
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    print(api_key[:3])
    
MODEL = os.getenv('GROK4F_MODEL')
openai = OpenAI(base_url=os.getenv('GROK_BASE_URL'), api_key=api_key)

API key looks good so far


In [19]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 '

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [20]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [21]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [22]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwar

In [23]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'press release',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'patent',
   'url': 'https://patents.google.com/patent/US20210049536A1/'},
  {'type': 'courses page',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'}]}

In [24]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [25]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling grok-4-fast
Found 3 relevant links
Found 3 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [26]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling grok-4-fast
Found 7 relevant links
Found 7 relevant links


{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [27]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [28]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling grok-4-fast
Found 2 relevant links
Found 2 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-M2
Updated
5 days ago
•
726k
•
977
deepseek-ai/DeepSeek-OCR
Updated
10 days ago
•
2.06M
•
2.4k
moonshotai/Kimi-Linear-48B-A3B-Instruct
Updated
2 days ago
•
15k
•
319
briaai/FIBO
Updated
about 23 hours ago
•
2.84k
•
191
dx8152/Qwen-Edit-2509-Multiple-angles
Updated
about 8 hours ago
•
154
Browse 1M+ models
Spaces
Running
on
CPU Upgrade
1.13k
1.13k
The Smol Training Playbook: The Secrets to Building World-Class LLMs
📝
Running
15.6k
15.6k
DeepSite v3
🐳
Generate any application by Vibe Coding
Runn

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [30]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [31]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling grok-4-fast
Found 6 relevant links
Found 6 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-M2\nUpdated\n5 days ago\n•\n726k\n•\n977\ndeepseek-ai/DeepSeek-OCR\nUpdated\n10 days ago\n•\n2.06M\n•\n2.4k\nmoonshotai/Kimi-Linear-48B-A3B-Instruct\nUpdated\n2 days ago\n•\n15k\n•\n319\nbriaai/FIBO\nUpdated\nabout 23 hours ago\n•\n2.84k\n•\n191\ndx8152/Qwen-Edit-2509-Multiple-angles\nUpdated\nabout 8 hours ago\n•\n154\nBrowse 1M+ models\nSpaces\nRunning\non\nCPU Upgrade\n

In [35]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [36]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling grok-4-fast
Found 3 relevant links
Found 3 relevant links


# Discover Hugging Face: Building the Future of AI Together

## Welcome to the AI Community
Hugging Face is the world's leading open platform where the machine learning (ML) community collaborates to create, discover, and deploy AI innovations. With over 1 million models, 250,000+ datasets, and 400,000+ applications (Spaces), we empower developers, researchers, and organizations to push the boundaries of AI. From text and image generation to video, audio, and even 3D modalities, Hugging Face is the home of ML—fostering open-source collaboration to accelerate real-world impact.

## Our Platform: Collaborate and Innovate
- **Models**: Browse and host 1M+ cutting-edge models, from trending ones like DeepSeek-OCR for optical character recognition to MiniMax-M2 for advanced language tasks. Share your work publicly or privately to build your ML portfolio.
- **Datasets**: Access 250k+ datasets, including specialized ones like NVIDIA's PhysicalAI for autonomous vehicles or fine-tuned collections for agent training.
- **Spaces**: Deploy and run 400k+ interactive AI apps effortlessly, with options for CPU, ZeroGPU, and more. Examples include video generation tools like Wan2.2 Animate and collaborative playbooks for LLM training.
  
Powered by the HF open-source stack, our platform enables seamless collaboration on unlimited public resources, helping you move faster and explore all AI modalities.

## Company Culture: Open, Collaborative, and Forward-Thinking
At Hugging Face, we embody a vibrant, community-driven culture rooted in openness and innovation. As pioneers in democratizing AI, we value collaboration, diversity, and ethical AI development. Our global team thrives on sharing knowledge through forums, blogs, and events, creating an inclusive environment where ideas flourish. We're committed to building tools that benefit everyone, from individual creators to enterprise giants, while prioritizing security, scalability, and sustainability.

## Serving Top Customers and Partners
Hugging Face powers AI for forward-thinking organizations worldwide, including leading tech companies like NVIDIA and innovative startups in autonomous vehicles, content generation, and more. Our customers range from solo developers showcasing portfolios to enterprise teams at scale—trusted by millions in the ML community. Whether you're a researcher fine-tuning models or a business deploying AI apps, we provide the infrastructure to innovate securely.

## Enterprise Solutions: Scale with Confidence
Elevate your AI workflow with our Team and Enterprise offerings:
- **Team Plan**: Starting at $20/user/month—ideal for growing teams with features like 5x ZeroGPU quota, private storage (1TB per member), analytics, and priority support.
- **Enterprise**: Custom contracts for advanced needs, including SSO integration, audit logs, regional data controls, resource groups, inference providers, and dedicated support. Enjoy granular security, managed billing, and boosted compute for high-performance AI.

Personal users can upgrade to PRO for $9/month, unlocking 10x private storage, 20x inference credits, and exclusive perks like Dev Mode Spaces.

## Careers: Join the AI Revolution
Hugging Face is hiring passionate talent to shape the future of ML. Explore current openings on our careers page, from engineering roles in model optimization to community management and enterprise sales. We offer opportunities to work on groundbreaking projects in a supportive, remote-friendly environment with competitive benefits. Build your career collaborating with top AI minds—apply today and contribute to open-source advancements that impact billions.

## Get Started Today
Sign up for free to explore models, datasets, and Spaces, or contact sales for enterprise demos. Together, let's build the future of AI.  
[Visit Hugging Face](https://huggingface.co) | [Careers](https://huggingface.co/careers) | [Enterprise Hub](https://huggingface.co/enterprise)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [45]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [46]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling grok-4-fast
Found 6 relevant links


# Discover Hugging Face: Building the Future of AI Together

## Who We Are
Hugging Face is the leading collaboration platform for the machine learning (ML) community, empowering creators, engineers, scientists, and end users to build an open and ethical AI future. As the home of open-source ML, we provide a central hub to share, explore, discover, and experiment with models, datasets, and applications. Our mission is to accelerate innovation across all modalities—text, image, video, audio, and even 3D—while fostering a vibrant, inclusive community that drives the next generation of AI.

With over 1 million models, 400,000+ applications (Spaces), and 250,000+ datasets, Hugging Face is where the world's ML ecosystem thrives. We offer an open-source stack to help teams move faster, build portfolios, and collaborate seamlessly on public or private projects.

## Our Platform: Create, Collaborate, Innovate
- **Models & Datasets**: Browse and contribute to trending resources like MiniMax-M2 for advanced AI tasks, DeepSeek-OCR for optical character recognition, and specialized datasets from partners like NVIDIA for autonomous vehicles and physical AI.
- **Spaces**: Deploy and run interactive AI apps effortlessly—explore demos like video generation tools (Wan2.2 Animate) or vibe-based app creators (DeepSite v3), powered by CPU, GPU, or ZeroGPU for scalable performance.
- **Community Focus**: Host unlimited public repositories, share work globally, and build your ML profile. Our platform supports experimentation and discovery, making ML accessible to all.

## Enterprise Solutions: Scale Securely
For organizations ready to harness AI at scale, Hugging Face offers Team and Enterprise plans. Starting at $20/user/month for Team, or custom Enterprise options via sales contact.

Key Features:
- **Security & Control**: Single Sign-On (SSO), audit logs, granular access via resource groups, token management, and advanced security policies.
- **Performance & Scalability**: ZeroGPU quota boosts (5x for teams), advanced compute options, private storage (1 TB per member + $25/month per extra TB), and inference providers with usage analytics and spending limits.
- **Collaboration Tools**: Private datasets viewer, regions for data location management, and priority support to maximize your AI initiatives.
- **Billing & Insights**: Managed billing, yearly commits, and a unified dashboard for repository analytics.

Join forward-thinking organizations leveraging Hugging Face for enterprise-grade AI development with dedicated support and enhanced privacy.

## Our Culture: Open, Collaborative, and Innovative
At Hugging Face, we embody a community-driven ethos rooted in openness and ethical AI. Our culture celebrates collaboration, where diverse talents from around the world contribute to shared progress. We prioritize learning, experimentation, and inclusivity, empowering users to innovate without barriers. As a fast-growing team passionate about ML, we foster an environment that values creativity, transparency, and real-world impact—building tools that democratize AI for everyone.

## Customers and Impact
Hugging Face serves a global ML community, including top innovators like NVIDIA, DeepSeek, and Moonshot AI, who rely on our platform for cutting-edge models and datasets. Enterprises across industries use our solutions to accelerate AI adoption, from autonomous vehicles to generative applications. Whether you're a startup prototyping apps or a large organization scaling secure AI infrastructure, we're the trusted partner for collaborative ML excellence.

## Careers: Join the AI Revolution
We're always hiring passionate individuals to shape the future of AI. Explore current openings in engineering, research, product, and community roles at [Hugging Face Careers](https://huggingface.co/careers). Join a dynamic team that's redefining ML—contribute to open-source projects, collaborate with global experts, and grow your career in a supportive, innovative environment. No matter your background, if you're excited about ethical AI, we want you on board.

## Get Started Today
Sign up for free to explore our ecosystem, or contact sales for Enterprise solutions. Together, let's build the future of AI.

*Hugging Face: The AI community building the future.*  
[Models](https://huggingface.co/models) | [Datasets](https://huggingface.co/datasets) | [Spaces](https://huggingface.co/spaces) | [Enterprise](https://huggingface.co/enterprise)

In [47]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling grok-4-fast
Found 6 relevant links


# Discover Hugging Face: The AI Community Building the Future

## Who We Are
Hugging Face is the world's leading open-source platform for the machine learning (ML) community. We empower creators, developers, and organizations to collaborate on AI models, datasets, and applications—accelerating innovation across text, image, video, audio, and even 3D modalities. With over 1 million models, 400,000+ applications (Spaces), and 250,000+ datasets, we're the home of ML where anyone can create, discover, and share groundbreaking work. Our open-source stack helps teams move faster, build portfolios, and explore the frontiers of AI.

## Our Platform: Collaborate and Innovate
- **Models & Datasets**: Browse and contribute to trending resources like MiniMax-M2 (726k downloads) or NVIDIA's PhysicalAI-Autonomous-Vehicles dataset. Host unlimited public repositories for seamless collaboration.
- **Spaces**: Deploy interactive AI apps effortlessly— from video generation tools like Wan2.2 Animate to training playbooks running on ZeroGPU for scalable performance.
- **Community**: Join a vibrant ecosystem of ML enthusiasts. Share your work, build your profile, and collaborate on real-world applications.
- **Tools & Resources**: Access docs, pricing, and enterprise-grade features to accelerate your ML journey.

We foster a culture of openness, collaboration, and rapid iteration, making AI accessible to all—from hobbyists to global teams.

## For Enterprises: Scale AI with Confidence
Power your organization with Hugging Face's Team and Enterprise solutions, trusted by forward-thinking AI leaders.

- **Team Plan**: Starting at $20/user/month—includes advanced security, analytics, and 5x ZeroGPU quota for Spaces.
- **Enterprise Options**: Contact sales for custom contracts with SSO integration, audit logs, private storage (1TB+ per user), resource groups, token management, inference providers, and priority support.
- **Key Benefits**: Granular access controls, regional data management, billing oversight, and enhanced compute for secure, scalable AI development.

Whether you're building autonomous vehicles, generative apps, or enterprise AI, our platform ensures compliance, efficiency, and innovation.

## Our Customers and Impact
Hugging Face serves a global community of developers, researchers, and enterprises like NVIDIA, DeepSeek, and Moonshot AI. From startups prototyping apps to Fortune 500 teams deploying production ML, we enable faster time-to-value and collaborative breakthroughs. Our users span industries, creating everything from OCR tools to video synthesis, driving the AI revolution forward.

## Join Our Team: Careers at Hugging Face
We're always growing and seeking passionate talent to shape the future of AI. Explore current openings in engineering, research, product, and community roles. Work in a dynamic, remote-friendly environment that values open-source contributions, creativity, and work-life balance. Build with cutting-edge tech, collaborate with top minds, and make a real impact—check our careers page for opportunities to join the Hugging Face family.

## Get Started Today
Sign up for free to explore models, launch a Space, or collaborate. For enterprise inquiries, [contact sales](https://huggingface.co/enterprise). Let's build the future of AI together!

*Hugging Face – Where ML Meets Community.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>